# Demo: 50 条英文问题翻译与意图解析

这个 notebook 顺序读取 `browsecomp_plus_hard50.jsonl` 中的 50 个问题，调用 vLLM OpenAI-compatible 服务，让模型输出：

- 原始英文问题
- 中文翻译
- 这段问题想问的是什么
- gold answer

本 notebook 不做检索、不调用 BM25，只用于快速理解题目。

## 1. 配置

In [ ]:
from pathlib import Path
import json
import re
import time
from typing import Any, Dict, List, Optional

import pandas as pd
from IPython.display import display

from agent.vllm_client import VLLMClient


def find_project_root() -> Path:
    cwd = Path.cwd()
    if (cwd / "browsecomp_plus_hard50.jsonl").exists():
        return cwd
    candidate = cwd / "HW3new" / "NLPproject"
    if (candidate / "browsecomp_plus_hard50.jsonl").exists():
        return candidate
    return Path(r"E:\CleanCodes\LearningField\NLP\HW3new\NLPproject")


project_root = find_project_root()
DATASET_PATH = project_root / "browsecomp_plus_hard50.jsonl"
OUTPUT_JSONL = project_root / "runs" / "demo_translation_intent.jsonl"
OUTPUT_CSV = project_root / "runs" / "demo_translation_intent.csv"

VLLM_BASE_URL = "http://127.0.0.1:8000/v1"
MODEL_NAME = "qwen_auto"
API_KEY = "dummy"

LIMIT = 50
TEMPERATURE = 0.0
MAX_TOKENS = 900
SLEEP_SECONDS_BETWEEN_CALLS = 0.0

client = VLLMClient(base_url=VLLM_BASE_URL, api_key=API_KEY)

print("project_root:", project_root)
print("dataset:", DATASET_PATH)
print("output_jsonl:", OUTPUT_JSONL)
print("model:", MODEL_NAME, "base_url:", VLLM_BASE_URL)


## 2. 数据读取与模型调用辅助函数

In [ ]:
def load_jsonl(path: Path, limit: Optional[int] = None) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            rows.append(json.loads(line))
            if limit is not None and len(rows) >= limit:
                break
    return rows


def qwen_extra_payload(model_name: str) -> Optional[Dict[str, Any]]:
    if "qwen" in str(model_name or "").lower():
        return {"chat_template_kwargs": {"enable_thinking": False}}
    return None


def strip_thinking(text: str) -> str:
    text = re.sub(r"<think>.*?</think>", "", str(text or ""), flags=re.S | re.I)
    return text.strip()


def extract_json_object(text: str) -> Optional[Dict[str, Any]]:
    text = strip_thinking(text)
    fenced = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, flags=re.S | re.I)
    candidates = [fenced.group(1)] if fenced else []
    candidates.append(text)
    decoder = json.JSONDecoder()
    for candidate in candidates:
        for match in re.finditer(r"\{", candidate):
            try:
                value, _ = decoder.raw_decode(candidate[match.start():])
            except json.JSONDecodeError:
                continue
            if isinstance(value, dict):
                return value
    return None


SYSTEM_PROMPT = """你是一个英文复杂问答题目解析助手。

任务：
1. 将英文问题忠实翻译成中文，保留所有数字、年份、专名、页码、引号内容和限定条件。
2. 用中文概括“这段问题想问的是什么”。这不是让你解题，而是说明最终要找的答案类型、目标实体/属性，以及关键限定链路。
3. 不要使用 gold answer 推断翻译或意图；gold answer 只会在程序输出表中展示。

输出必须是严格 JSON：
{
  "chinese_translation": "...",
  "question_intent": "..."
}
"""


def analyze_question_with_model(row: Dict[str, Any], max_retries: int = 2) -> Dict[str, str]:
    query_id = str(row.get("query_id", ""))
    question = row.get("query", "")
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (
                f"query_id: {query_id}\n\n"
                "English question:\n"
                f"{question}\n\n"
                "请只输出 JSON。"
            ),
        },
    ]

    last_error = ""
    for attempt in range(max_retries + 1):
        try:
            response = client.simple_chat(
                model=MODEL_NAME,
                messages=messages,
                temperature=TEMPERATURE,
                max_tokens=MAX_TOKENS,
                extra_payload=qwen_extra_payload(MODEL_NAME),
            )
            content = response["choices"][0]["message"].get("content", "")
            parsed = extract_json_object(content)
            if parsed:
                return {
                    "chinese_translation": str(parsed.get("chinese_translation", "")).strip(),
                    "question_intent": str(parsed.get("question_intent", "")).strip(),
                    "raw_model_output": content,
                    "parse_status": "ok",
                }
            last_error = "JSON parse failed"
            messages.append({"role": "assistant", "content": content})
            messages.append({"role": "user", "content": "上一次输出不是严格 JSON。请只输出包含 chinese_translation 和 question_intent 的 JSON。"})
        except Exception as exc:
            last_error = repr(exc)
            if attempt < max_retries:
                time.sleep(1.0 + attempt)

    return {
        "chinese_translation": "",
        "question_intent": "",
        "raw_model_output": last_error,
        "parse_status": "failed",
    }


rows = load_jsonl(DATASET_PATH, limit=LIMIT)
print(f"loaded rows: {len(rows)}")


## 3. 顺序处理 50 个问题

In [ ]:
OUTPUT_JSONL.parent.mkdir(parents=True, exist_ok=True)

records: List[Dict[str, Any]] = []
start_time = time.perf_counter()

with OUTPUT_JSONL.open("w", encoding="utf-8") as f:
    for index, row in enumerate(rows, start=1):
        qid = str(row.get("query_id", ""))
        print(f"[{index}/{len(rows)}] query_id={qid} START", flush=True)

        result = analyze_question_with_model(row)
        record = {
            "query_id": qid,
            "english_question": row.get("query", ""),
            "chinese_translation": result["chinese_translation"],
            "question_intent": result["question_intent"],
            "gold_answer": row.get("answer", ""),
            "parse_status": result["parse_status"],
            "raw_model_output": result["raw_model_output"],
        }
        records.append(record)
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
        f.flush()

        elapsed = time.perf_counter() - start_time
        print(
            f"[{index}/{len(rows)}] query_id={qid} DONE "
            f"elapsed={elapsed:.1f}s parse={record['parse_status']}",
            flush=True,
        )

        if SLEEP_SECONDS_BETWEEN_CALLS > 0:
            time.sleep(SLEEP_SECONDS_BETWEEN_CALLS)

df = pd.DataFrame(records)
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print(f"saved jsonl: {OUTPUT_JSONL}")
print(f"saved csv: {OUTPUT_CSV}")


## 4. 查看结果

In [ ]:
display_columns = [
    "query_id",
    "english_question",
    "chinese_translation",
    "question_intent",
    "gold_answer",
]

pd.set_option("display.max_colwidth", 240)
display(df[display_columns])


## 5. 仅查看解析失败项

In [ ]:
failed = df[df["parse_status"] != "ok"]
print("failed count:", len(failed))
if len(failed):
    display(failed[["query_id", "english_question", "raw_model_output"]])
